In [ ]:
# ═══════════════════════════════════════════════════════
# 0. 기본 설정
# ═══════════════════════════════════════════════════════
from pathlib import Path
import numpy as np
import pandas as pd
import os
import random
import re
from sklearn.impute import SimpleImputer
import lightgbm as lgb

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ═══════════════════════════════════════════════════════
# 1. 원본 데이터 로드
# ═══════════════════════════════════════════════════════
DATA_DIR = Path("/content/drive/MyDrive/비타민/풍력발전량예측/data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("train_labels:", train_labels.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)

train_labels: (26304, 4)
ldaps_train: (420864, 35) gfs_train: (236736, 40)


In [ ]:
# ═══════════════════════════════════════════════════════
# 2. info.xlsx 파싱 (터빈 메타 정보)
# ═══════════════════════════════════════════════════════
raw = pd.read_excel(DATA_DIR / "info.xlsx", sheet_name="info", header=None)
info = raw.iloc[4:21].copy()
info.columns = raw.iloc[3]
info = info.reset_index(drop=True)
info = info.rename(columns={
    "단계": "stage", "명칭": "site", "제작사": "maker", "모델명": "model", "호기": "turbine_no",
    "좌표(Google)": "coord", "KPX그룹": "kpx_group", "Hub Height(m)": "hub_height",
    "Rotor Diameter(m)": "rotor_diameter", "설비용량(MW)": "capacity_mw", "그룹설비용량(MW)": "group_capacity_mw",
})
info["kpx_group"] = info["kpx_group"].ffill().astype(int)


def dms_to_decimal(s):
    pattern = r"(\d+)°(\d+)'([\d.]+)\"([NS])\s+(\d+)°(\d+)'([\d.]+)\"([EW])"
    m = re.match(pattern, s.strip())
    if not m:
        raise ValueError(f"좌표 파싱 실패: {s}")
    d1, m1, s1, hemi1, d2, m2, s2, hemi2 = m.groups()
    lat = float(d1) + float(m1) / 60 + float(s1) / 3600
    if hemi1 == "S": lat = -lat
    lon = float(d2) + float(m2) / 60 + float(s2) / 3600
    if hemi2 == "W": lon = -lon
    return lat, lon


coords = info["coord"].apply(dms_to_decimal)
info["lat"] = coords.apply(lambda x: x[0])
info["lon"] = coords.apply(lambda x: x[1])
assert info.groupby("kpx_group")["capacity_mw"].sum().tolist() == [21.6, 21.6, 21.0]

group_coords = info.groupby("kpx_group")[["lat", "lon"]].mean().reset_index()
group_coords = group_coords.rename(columns={"kpx_group": "KPX그룹"})


/tmp/ipykernel_676/3448643582.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  info["kpx_group"] = info["kpx_group"].ffill().astype(int)


In [ ]:
# ═══════════════════════════════════════════════════════
# 3. 거리 계산 & grid 매칭
# ═══════════════════════════════════════════════════════
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def find_nearest_grid(weather_df, coord_df, name_col, lat_col="lat", lon_col="lon"):
    grids = weather_df[["grid_id", "latitude", "longitude"]].drop_duplicates()
    mapping = {}
    for _, row in coord_df.iterrows():
        dists = haversine(row[lat_col], row[lon_col], grids["latitude"], grids["longitude"])
        mapping[row[name_col]] = grids.loc[dists.idxmin(), "grid_id"]
    return mapping


ldaps_group_grid = find_nearest_grid(ldaps_train, group_coords, "KPX그룹")
gfs_group_grid = find_nearest_grid(gfs_train, group_coords, "KPX그룹")
print("LDAPS 매칭:", ldaps_group_grid)
print("GFS 매칭:", gfs_group_grid)

LDAPS 매칭: {np.float64(1.0): np.int64(5), np.float64(2.0): np.int64(6), np.float64(3.0): np.int64(12)}
GFS 매칭: {np.float64(1.0): np.int64(5), np.float64(2.0): np.int64(5), np.float64(3.0): np.int64(5)}


In [ ]:
# ═══════════════════════════════════════════════════════
# 4. 풍속/풍향/윈드시어 명시적 계산
# ═══════════════════════════════════════════════════════
def add_wind_features(df, uv_pairs):
    df = df.copy()
    for u_col, v_col, height in uv_pairs:
        df[f"ws_{height}"] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        wd_rad = np.arctan2(df[v_col], df[u_col])
        df[f"wd_sin_{height}"] = np.sin(wd_rad)
        df[f"wd_cos_{height}"] = np.cos(wd_rad)
    return df


LDAPS_UV_PAIRS = [("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m")]
GFS_UV_PAIRS = [
    ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m"),
    ("heightAboveGround_80_u", "heightAboveGround_80_v", "80m"),
    ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "100m"),
    ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl"),
]

ldaps_train = add_wind_features(ldaps_train, LDAPS_UV_PAIRS)
ldaps_test = add_wind_features(ldaps_test, LDAPS_UV_PAIRS)
gfs_train = add_wind_features(gfs_train, GFS_UV_PAIRS)
gfs_test = add_wind_features(gfs_test, GFS_UV_PAIRS)
for df in [gfs_train, gfs_test]:
    df["wind_shear_100_10"] = df["ws_100m"] - df["ws_10m"]
    df["wind_shear_80_10"] = df["ws_80m"] - df["ws_10m"]

In [ ]:
# ═══════════════════════════════════════════════════════
# 5. 그룹별 기상 피처 생성 (centroid 1개 grid)
# ═══════════════════════════════════════════════════════
def build_group_weather(weather_df, group_grid_map, prefix):
    weather_df = weather_df.copy()
    weather_df["forecast_kst_dtm"] = pd.to_datetime(weather_df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in weather_df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    frames = []
    for group, grid_id in group_grid_map.items():
        sub = weather_df.loc[weather_df["grid_id"] == grid_id, ["forecast_kst_dtm", *value_cols]].copy()
        sub = sub.rename(columns={c: f"{prefix}_{c}" for c in value_cols})
        sub["kpx_group"] = int(group)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)


ldaps_group_train = build_group_weather(ldaps_train, ldaps_group_grid, "ldaps")
ldaps_group_test = build_group_weather(ldaps_test, ldaps_group_grid, "ldaps")
gfs_group_train = build_group_weather(gfs_train, gfs_group_grid, "gfs")
gfs_group_test = build_group_weather(gfs_test, gfs_group_grid, "gfs")

In [ ]:
# ═══════════════════════════════════════════════════════
# 6. train_labels wide → long + v5 데이터셋 조립
# ═══════════════════════════════════════════════════════
train_labels_long = train_labels.melt(
    id_vars=["kst_dtm"], value_vars=TARGET_COLS, var_name="kpx_group_col", value_name="target",
)
train_labels_long["kpx_group"] = train_labels_long["kpx_group_col"].str.extract(r"(\d+)").astype(int)
train_labels_long = train_labels_long.rename(columns={"kst_dtm": "forecast_kst_dtm"}).drop(columns=["kpx_group_col"])

test_base = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
test_long = pd.concat([test_base.assign(kpx_group=g) for g in [1, 2, 3]], ignore_index=True)

train_df_v5 = train_labels_long.merge(
    ldaps_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left"
).merge(gfs_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left")

test_df_v5 = test_long.merge(
    ldaps_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left"
).merge(gfs_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left")

assert train_df_v5.shape[0] == train_labels_long.shape[0]
weather_cols_v5 = [c for c in train_df_v5.columns if c not in ["forecast_kst_dtm", "kpx_group", "target"]]


In [ ]:
# ═══════════════════════════════════════════════════════
# 7. 검증 공통 함수
# ═══════════════════════════════════════════════════════
def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out


def metric(answer_df, pred_df):
    group_nmae, group_ficr = [], []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]
        valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    one_minus_nmae = 1 - np.mean(group_nmae)
    ficr = np.mean(group_ficr)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr


def metric_single_group(actual, forecast, capacity):
    actual, forecast = np.asarray(actual, dtype=float), np.asarray(forecast, dtype=float)
    valid = actual >= capacity * 0.10
    actual_v, forecast_v = actual[valid], forecast[valid]
    if len(actual_v) == 0:
        return np.nan, np.nan, 0
    error_rate = np.abs(forecast_v - actual_v) / capacity
    nmae = np.mean(error_rate)
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr = np.sum(actual_v * unit_price) / np.sum(actual_v * 4.0)
    return nmae, ficr, len(actual_v)


def make_sample_weight(y, capacity, low_weight=0.05):
    threshold = capacity * 0.10
    return np.where(y >= threshold, 1.0, low_weight)


def get_lgbm_objective(objective_name, alpha=None):
    params = dict(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        num_leaves=31, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=1, verbose=-1, objective=objective_name,
    )
    if alpha is not None:
        params["alpha"] = alpha
    return lgb.LGBMRegressor(**params)


def apply_rolling_smoothing(pred_series, window=3, alpha=0.5):
    rolling = pred_series.rolling(window, center=True, min_periods=1).mean()
    return alpha * pred_series + (1 - alpha) * rolling


def apply_bias_correction_v2(pred, ws, group, bias_map):
    pred = pred.copy()
    if group not in bias_map:
        return pred
    for interval, bias in bias_map[group].items():
        mask = (ws > interval.left) & (ws <= interval.right)
        pred[mask] = pred[mask] * (1 - bias)
    return pred

In [ ]:
# ═══════════════════════════════════════════════════════
# 8. 예보 불일치도(v9) + 계절 인터랙션(v10) 피처
# ═══════════════════════════════════════════════════════
def add_forecast_disagreement(df):
    df = df.copy()
    df["ws_disagreement"] = np.abs(df["ldaps_ws_10m"] - df["gfs_ws_10m"])
    df["wd_agreement"] = df["ldaps_wd_cos_10m"] * df["gfs_wd_cos_10m"] + df["ldaps_wd_sin_10m"] * df["gfs_wd_sin_10m"]
    return df


def add_seasonal_interaction(df):
    df = df.copy()
    month = pd.to_datetime(df["forecast_kst_dtm"]).dt.month
    df["high_variance_season"] = month.isin([1, 2, 3, 4, 5, 7]).astype(int)
    df["ws_x_season"] = df["ldaps_ws_10m"] * df["high_variance_season"]
    df["disagreement_x_season"] = df["ws_disagreement"] * df["high_variance_season"]
    return df


train_df_v9 = add_forecast_disagreement(train_df_v5)
test_df_v9 = add_forecast_disagreement(test_df_v5)
train_df_v10 = add_seasonal_interaction(train_df_v9)
test_df_v10 = add_seasonal_interaction(test_df_v9)

weather_cols_v10 = [c for c in train_df_v10.columns if c not in ["forecast_kst_dtm", "kpx_group", "target"]]
print(f"v10 피처 개수: {len(weather_cols_v10)}")

v10 피처 개수: 87


In [ ]:
# ═══════════════════════════════════════════════════════
# 9. 새 검증 기준: 2024년 분기별 4-fold
# ═══════════════════════════════════════════════════════
FOLDS_Q = [
    {"name": "fold_2024Q1", "train_end": pd.Timestamp("2024-01-01 00:00:00"),
     "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2024-04-01 00:00:00")},
    {"name": "fold_2024Q2", "train_end": pd.Timestamp("2024-04-01 00:00:00"),
     "valid_start": pd.Timestamp("2024-04-01 01:00:00"), "valid_end": pd.Timestamp("2024-07-01 00:00:00")},
    {"name": "fold_2024Q3", "train_end": pd.Timestamp("2024-07-01 00:00:00"),
     "valid_start": pd.Timestamp("2024-07-01 01:00:00"), "valid_end": pd.Timestamp("2024-10-01 00:00:00")},
    {"name": "fold_2024Q4", "train_end": pd.Timestamp("2024-10-01 00:00:00"),
     "valid_start": pd.Timestamp("2024-10-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01 00:00:00")},
]


def estimate_bias_correction_adaptive(train_df, weather_cols, tr_end, low_weight=0.05,
                                        objective="mae", holdout_ratio=0.25, min_fit_days=90):
    """고정 180일 대신, 그룹별 가용 데이터 기간의 holdout_ratio만큼만 held-out (최소 min_fit_days 보장)"""
    bias_records = []
    for group in [1, 2, 3]:
        target_col = f"kpx_group_{group}"
        capacity = CAPACITY_KWH[target_col]

        g_data = train_df[(train_df["kpx_group"] == group) & (train_df["forecast_kst_dtm"] <= tr_end)].copy()
        g_data = g_data[g_data["target"].notna()]
        if len(g_data) == 0:
            continue

        first_date = g_data["forecast_kst_dtm"].min()
        available_days = max((tr_end - first_date).days, 1)
        holdout_days = min(int(available_days * holdout_ratio), 180)
        if available_days - holdout_days < min_fit_days:
            holdout_days = max(0, available_days - min_fit_days)

        tr_sub_end = tr_end - pd.Timedelta(days=holdout_days)
        fit_g = g_data[g_data["forecast_kst_dtm"] <= tr_sub_end]
        check_g = g_data[g_data["forecast_kst_dtm"] > tr_sub_end]
        if len(fit_g) == 0 or len(check_g) == 0:
            continue

        fit_feat = pd.concat([calendar_features(fit_g["forecast_kst_dtm"]).reset_index(drop=True), fit_g[weather_cols].reset_index(drop=True)], axis=1)
        check_feat = pd.concat([calendar_features(check_g["forecast_kst_dtm"]).reset_index(drop=True), check_g[weather_cols].reset_index(drop=True)], axis=1)

        imputer = SimpleImputer(strategy="median")
        X_fit = pd.DataFrame(imputer.fit_transform(fit_feat), columns=fit_feat.columns)
        X_check = pd.DataFrame(imputer.transform(check_feat), columns=check_feat.columns)
        y_fit = fit_g["target"]
        w_fit = make_sample_weight(y_fit.to_numpy(), capacity, low_weight=low_weight)

        model = get_lgbm_objective(objective)
        model.fit(X_fit, y_fit, sample_weight=w_fit)
        pred_check = model.predict(X_check)

        check_g = check_g.copy()
        check_g["signed_error_rate"] = (pred_check - check_g["target"]) / capacity
        valid = check_g["target"] >= capacity * 0.10
        check_g = check_g.loc[valid]
        if len(check_g) == 0:
            continue

        check_g["ws_bin"] = pd.cut(check_g["ldaps_ws_10m"], bins=[0, 11, 13, 100])
        bias_by_bin = check_g.groupby("ws_bin", observed=True)["signed_error_rate"].mean()
        bias_records.append({"group": group, "bias_by_bin": bias_by_bin.to_dict()})
        print(f"  그룹{group}: 가용 {available_days}일 → held-out {holdout_days}일 (fit {len(fit_g)}행, check {len(check_g)}행)")

    return bias_records

  그룹1: 가용 729일 → held-out 180일 (fit 13105행, check 2634행)
  그룹2: 가용 729일 → held-out 180일 (fit 13105행, check 2643행)
  그룹3: 가용 364일 → held-out 91일 (fit 6576행, check 1458행)
[fold_2024Q1] Score=0.6099 | 1-NMAE=0.8606 | FICR=0.3593
  그룹1: 가용 820일 → held-out 180일 (fit 15286행, check 2915행)
  그룹2: 가용 820일 → held-out 180일 (fit 15287행, check 2908행)
  그룹3: 가용 455일 → held-out 113일 (fit 8232행, check 1624행)
[fold_2024Q2] Score=0.6290 | 1-NMAE=0.8830 | FICR=0.3750
  그룹1: 가용 911일 → held-out 180일 (fit 17470행, check 2516행)
  그룹2: 가용 911일 → held-out 180일 (fit 17471행, check 2551행)
  그룹3: 가용 546일 → held-out 136일 (fit 9864행, check 1706행)
[fold_2024Q3] Score=0.6372 | 1-NMAE=0.8817 | FICR=0.3927
  그룹1: 가용 1003일 → held-out 180일 (fit 19678행, check 2284행)
  그룹2: 가용 1003일 → held-out 180일 (fit 19679행, check 2290행)
  그룹3: 가용 638일 → held-out 159일 (fit 11520행, check 1813행)
[fold_2024Q4] Score=0.6253 | 1-NMAE=0.8808 | FICR=0.3697

=== 방식 A: fold 단순평균 ===
score     0.625354
1-NMAE    0.876518
FICR      0.374191
dtype: f

## 앙상블

In [ ]:
# ═══════════════════════════════════════════════════════
# 12. LightGBM + XGBoost 블렌딩 (4-fold 체계)
# ═══════════════════════════════════════════════════════
import xgboost as xgb


def run_cv_blend_v2(train_df, weather_cols, folds, low_weight=0.05, n_seeds=5,
                     roll_window=3, roll_alpha=0.7, holdout_ratio=0.25, blend_weight=0.5):
    """
    blend_weight: LightGBM 비중 (1-blend_weight가 XGBoost 비중)
    편향보정/rolling/하한clip은 블렌딩된 예측값에 적용
    """
    fold_scores, group_scores = [], []
    all_answer, all_pred = [], []

    for fold in folds:
        bias_estimates = estimate_bias_correction_adaptive(
            train_df, weather_cols, fold["train_end"], low_weight, "mae", holdout_ratio
        )
        bias_map = {b["group"]: b["bias_by_bin"] for b in bias_estimates}

        tr_mask = train_df["forecast_kst_dtm"] <= fold["train_end"]
        va_mask = (train_df["forecast_kst_dtm"] >= fold["valid_start"]) & (train_df["forecast_kst_dtm"] <= fold["valid_end"])
        tr_all = train_df.loc[tr_mask].reset_index(drop=True)
        va_all = train_df.loc[va_mask].reset_index(drop=True)

        va_time_index = sorted(va_all["forecast_kst_dtm"].unique())
        answer_df = pd.DataFrame({"forecast_kst_dtm": va_time_index})
        pred_df = pd.DataFrame({"forecast_kst_dtm": va_time_index})

        for group in [1, 2, 3]:
            target_col = f"kpx_group_{group}"
            capacity = CAPACITY_KWH[target_col]
            tr_g = tr_all.loc[tr_all["kpx_group"] == group].copy()
            va_g = va_all.loc[va_all["kpx_group"] == group].sort_values("forecast_kst_dtm").copy()

            tr_feat = pd.concat([calendar_features(tr_g["forecast_kst_dtm"]).reset_index(drop=True), tr_g[weather_cols].reset_index(drop=True)], axis=1)
            va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True), va_g[weather_cols].reset_index(drop=True)], axis=1)

            target_mask = tr_g["target"].notna().to_numpy()
            va_answer = va_g[["forecast_kst_dtm", "target"]].rename(columns={"target": target_col})
            answer_df = answer_df.merge(va_answer, on="forecast_kst_dtm", how="left")
            if target_mask.sum() == 0:
                pred_df[target_col] = 0.0
                group_scores.append({"fold": fold["name"], "group": group, "1-NMAE": np.nan, "FICR": np.nan, "valid_n": 0})
                continue

            imputer = SimpleImputer(strategy="median")
            X_tr = pd.DataFrame(imputer.fit_transform(tr_feat.loc[target_mask]), columns=tr_feat.columns)
            X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
            y_tr = tr_g.loc[target_mask, "target"]
            w_tr = make_sample_weight(y_tr.to_numpy(), capacity, low_weight=low_weight)

            # ── LightGBM bagging ──
            lgbm_preds = []
            for seed in range(n_seeds):
                model = lgb.LGBMRegressor(
                    n_estimators=500, max_depth=6, learning_rate=0.03,
                    num_leaves=31, min_child_samples=20,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=seed, n_jobs=1, verbose=-1, objective="mae",
                )
                model.fit(X_tr, y_tr, sample_weight=w_tr)
                lgbm_preds.append(model.predict(X_va))
            pred_lgbm = np.mean(lgbm_preds, axis=0)

            # ── XGBoost bagging ──
            xgb_preds = []
            for seed in range(n_seeds):
                model = xgb.XGBRegressor(
                    n_estimators=500, max_depth=6, learning_rate=0.03,
                    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                    random_state=seed, n_jobs=1, verbosity=0, objective="reg:absoluteerror",
                )
                model.fit(X_tr, y_tr, sample_weight=w_tr)
                xgb_preds.append(model.predict(X_va))
            pred_xgb = np.mean(xgb_preds, axis=0)

            # ── 블렌딩 ──
            pred = blend_weight * pred_lgbm + (1 - blend_weight) * pred_xgb

            # ── 편향보정 + rolling + 하한clip ──
            pred = apply_bias_correction_v2(pred, va_g["ldaps_ws_10m"].to_numpy(), group, bias_map)
            pred_series = pd.Series(pred, index=va_g.index)
            pred_series = apply_rolling_smoothing(pred_series, window=roll_window, alpha=roll_alpha)
            pred = pred_series.to_numpy()
            pred = np.clip(pred, capacity * 0.10, capacity)

            va_pred = va_g[["forecast_kst_dtm"]].copy()
            va_pred[target_col] = pred
            pred_df = pred_df.merge(va_pred, on="forecast_kst_dtm", how="left")

            nmae_g, ficr_g, valid_n = metric_single_group(va_g["target"].to_numpy(), pred, capacity)
            group_scores.append({
                "fold": fold["name"], "group": group,
                "1-NMAE": 1 - nmae_g if not np.isnan(nmae_g) else np.nan,
                "FICR": ficr_g, "valid_n": valid_n,
            })

        answer_df_f = answer_df[TARGET_COLS].fillna(0.0)
        pred_df_f = pred_df[TARGET_COLS].fillna(0.0)
        score, nmae_score, ficr_score = metric(answer_df_f, pred_df_f)
        fold_scores.append({"fold": fold["name"], "score": score, "1-NMAE": nmae_score, "FICR": ficr_score})
        print(f"[{fold['name']}] Score={score:.4f} | 1-NMAE={nmae_score:.4f} | FICR={ficr_score:.4f}")

        all_answer.append(answer_df_f)
        all_pred.append(pred_df_f)

    cv_result = pd.DataFrame(fold_scores)
    group_result = pd.DataFrame(group_scores)

    print("\n=== 방식 A: fold 단순평균 ===")
    print(cv_result[["score", "1-NMAE", "FICR"]].mean())
    print("표준편차(score):", cv_result["score"].std())

    pooled_answer = pd.concat(all_answer, ignore_index=True)
    pooled_pred = pd.concat(all_pred, ignore_index=True)
    pooled_score, pooled_nmae, pooled_ficr = metric(pooled_answer, pooled_pred)
    print("\n=== 방식 B: Pooled ===")
    print(f"Score={pooled_score:.4f} | 1-NMAE={pooled_nmae:.4f} | FICR={pooled_ficr:.4f}")

    print("\n=== 그룹별 평균 ===")
    print(group_result.groupby("group")[["1-NMAE", "FICR"]].mean())

    return cv_result, group_result, (pooled_score, pooled_nmae, pooled_ficr)


# ── 블렌딩 비율 몇 가지 실험 ──
print("########## 블렌딩 50:50 ##########")
cv_blend50, group_blend50, pooled_blend50 = run_cv_blend_v2(
    train_df_v10, weather_cols_v10, FOLDS_Q, n_seeds=5, blend_weight=0.5
)

print("\n########## 블렌딩 70:30 (lgbm 비중↑) ##########")
cv_blend70, group_blend70, pooled_blend70 = run_cv_blend_v2(
    train_df_v10, weather_cols_v10, FOLDS_Q, n_seeds=5, blend_weight=0.7
)

print("\n########## 최종 비교 ##########")
print("v10 단독(LightGBM만, 기존 최선):", pooled_q[0])
print("블렌딩 50:50:                    ", pooled_blend50[0])
print("블렌딩 70:30:                    ", pooled_blend70[0])

########## 블렌딩 50:50 ##########
  그룹1: 가용 729일 → held-out 180일 (fit 13105행, check 2634행)
  그룹2: 가용 729일 → held-out 180일 (fit 13105행, check 2643행)
  그룹3: 가용 364일 → held-out 91일 (fit 6576행, check 1458행)
[fold_2024Q1] Score=0.6108 | 1-NMAE=0.8612 | FICR=0.3604
  그룹1: 가용 820일 → held-out 180일 (fit 15286행, check 2915행)
  그룹2: 가용 820일 → held-out 180일 (fit 15287행, check 2908행)
  그룹3: 가용 455일 → held-out 113일 (fit 8232행, check 1624행)
[fold_2024Q2] Score=0.6294 | 1-NMAE=0.8830 | FICR=0.3758
  그룹1: 가용 911일 → held-out 180일 (fit 17470행, check 2516행)
  그룹2: 가용 911일 → held-out 180일 (fit 17471행, check 2551행)
  그룹3: 가용 546일 → held-out 136일 (fit 9864행, check 1706행)
[fold_2024Q3] Score=0.6360 | 1-NMAE=0.8824 | FICR=0.3896
  그룹1: 가용 1003일 → held-out 180일 (fit 19678행, check 2284행)
  그룹2: 가용 1003일 → held-out 180일 (fit 19679행, check 2290행)
  그룹3: 가용 638일 → held-out 159일 (fit 11520행, check 1813행)
[fold_2024Q4] Score=0.6272 | 1-NMAE=0.8812 | FICR=0.3732

=== 방식 A: fold 단순평균 ===
score     0.625858
1-NMAE    0.87

In [ ]:
# ═══════════════════════════════════════════════════════
# 최종 학습: v10 + LightGBM/XGBoost 50:50 블렌딩 + 편향보정 + rolling → 제출
# ═══════════════════════════════════════════════════════
import xgboost as xgb

N_SEEDS = 5
LOW_WEIGHT = 0.05
ROLL_WINDOW, ROLL_ALPHA = 3, 0.7
HOLDOUT_RATIO = 0.25
BLEND_WEIGHT = 0.5  # LightGBM 비중

max_date = train_df_v10["forecast_kst_dtm"].max()
print("최종 편향 계수 추정 중...")
bias_estimates_final = estimate_bias_correction_adaptive(
    train_df_v10, weather_cols_v10, max_date, LOW_WEIGHT, "mae", HOLDOUT_RATIO
)
bias_map_final = {b["group"]: b["bias_by_bin"] for b in bias_estimates_final}
print("최종 편향 보정 계수:", bias_map_final)

predictions_final = pd.DataFrame(index=test_df_v10.index)

for group in [1, 2, 3]:
    target_col = f"kpx_group_{group}"
    capacity = CAPACITY_KWH[target_col]
    group_mask_tr = train_df_v10["kpx_group"] == group
    group_mask_te = test_df_v10["kpx_group"] == group

    target_mask = group_mask_tr & train_df_v10["target"].notna()
    y_train = train_df_v10.loc[target_mask, "target"]

    tr_feat = pd.concat([
        calendar_features(train_df_v10.loc[target_mask, "forecast_kst_dtm"]).reset_index(drop=True),
        train_df_v10.loc[target_mask, weather_cols_v10].reset_index(drop=True),
    ], axis=1)
    te_g = test_df_v10.loc[group_mask_te].sort_values("forecast_kst_dtm").copy()
    te_feat = pd.concat([
        calendar_features(te_g["forecast_kst_dtm"]).reset_index(drop=True),
        te_g[weather_cols_v10].reset_index(drop=True),
    ], axis=1)

    imputer = SimpleImputer(strategy="median")
    X_tr_imp = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_te_imp = pd.DataFrame(imputer.transform(te_feat), columns=te_feat.columns)

    w_train = make_sample_weight(y_train.to_numpy(), capacity, low_weight=LOW_WEIGHT)

    # ── LightGBM bagging ──
    lgbm_preds = []
    for seed in range(N_SEEDS):
        model = lgb.LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.03,
            num_leaves=31, min_child_samples=20,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, n_jobs=1, verbose=-1, objective="mae",
        )
        model.fit(X_tr_imp, y_train, sample_weight=w_train)
        lgbm_preds.append(model.predict(X_te_imp))
    pred_lgbm = np.mean(lgbm_preds, axis=0)

    # ── XGBoost bagging ──
    xgb_preds = []
    for seed in range(N_SEEDS):
        model = xgb.XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.03,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            random_state=seed, n_jobs=1, verbosity=0, objective="reg:absoluteerror",
        )
        model.fit(X_tr_imp, y_train, sample_weight=w_train)
        xgb_preds.append(model.predict(X_te_imp))
    pred_xgb = np.mean(xgb_preds, axis=0)

    # ── 블렌딩 50:50 ──
    pred = BLEND_WEIGHT * pred_lgbm + (1 - BLEND_WEIGHT) * pred_xgb

    # ── 편향보정 + rolling + 하한clip ──
    pred = apply_bias_correction_v2(pred, te_g["ldaps_ws_10m"].to_numpy(), group, bias_map_final)
    pred_series = pd.Series(pred, index=te_g.index)
    pred_series = apply_rolling_smoothing(pred_series, window=ROLL_WINDOW, alpha=ROLL_ALPHA)
    pred = np.clip(pred_series.to_numpy(), capacity * 0.10, capacity)

    predictions_final.loc[te_g.index, target_col] = pred
    print(target_col, "train rows:", int(target_mask.sum()))

# ── long → wide pivot ──
test_pred_long_final = test_df_v10[["forecast_id", "forecast_kst_dtm", "kpx_group"]].copy()
for group in [1, 2, 3]:
    target_col = f"kpx_group_{group}"
    mask = test_df_v10["kpx_group"] == group
    test_pred_long_final.loc[mask, "pred"] = predictions_final.loc[mask, target_col]

submission_final = test_pred_long_final.pivot(index=["forecast_id", "forecast_kst_dtm"], columns="kpx_group", values="pred")
submission_final.columns = [f"kpx_group_{int(c)}" for c in submission_final.columns]
submission_final = submission_final.reset_index()
submission_final["forecast_kst_dtm"] = pd.to_datetime(submission_final["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
submission_final = sample_submission[["forecast_id"]].merge(submission_final, on="forecast_id", how="left")

print(submission_final.head())
assert submission_final.shape[0] == sample_submission.shape[0]
assert submission_final[TARGET_COLS].isna().sum().sum() == 0
for col in TARGET_COLS:
    print(col, "max:", submission_final[col].max(), "/ capacity:", CAPACITY_KWH[col])
print("forecast_id 순서 일치:", (submission_final["forecast_id"] == sample_submission["forecast_id"]).all())

output_path = DATA_DIR / "v_blend5050_submit.csv"
submission_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")

최종 편향 계수 추정 중...
  그룹1: 가용 1095일 → held-out 180일 (fit 21886행, check 2376행)
  그룹2: 가용 1095일 → held-out 180일 (fit 21887행, check 2332행)
  그룹3: 가용 730일 → held-out 180일 (fit 13224행, check 2101행)
최종 편향 보정 계수: {1: {Interval(0, 11, closed='right'): -0.04228313814258492, Interval(11, 13, closed='right'): -0.02184178840020315, Interval(13, 100, closed='right'): -0.07065279829890429}, 2: {Interval(0, 11, closed='right'): -0.021582009147552265, Interval(11, 13, closed='right'): -0.04207017019116299, Interval(13, 100, closed='right'): 0.007923836095310529}, 3: {Interval(0, 11, closed='right'): -0.022869280724573306, Interval(11, 13, closed='right'): -0.07401785431732325, Interval(13, 100, closed='right'): -0.168794016172195}}
kpx_group_1 train rows: 26200
kpx_group_2 train rows: 26201
kpx_group_3 train rows: 17538
     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  19097.936437  18351.107568   
1  forecast_0002  2025-01-01 02:00:00  17568.6642